In [1]:
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt

Energy computation for single electron system (i.e., Hydrogen)

In [2]:
def hydrogen_energy(n):
    return -1.0 / (2.0 * n**2)
 
print("="*60)
print("HYDROGEN ATOM ENERGY LEVELS")
print("="*60)
for n in range(1, 6):
    E_hartree = hydrogen_energy(n)
    E_eV = E_hartree * 27.211
    print(f"n = {n}: E = {E_hartree:.6f} Hartree = {E_eV:.3f} eV")

HYDROGEN ATOM ENERGY LEVELS
n = 1: E = -0.500000 Hartree = -13.605 eV
n = 2: E = -0.125000 Hartree = -3.401 eV
n = 3: E = -0.055556 Hartree = -1.512 eV
n = 4: E = -0.031250 Hartree = -0.850 eV
n = 5: E = -0.020000 Hartree = -0.544 eV


Many Electron Problem

In [ ]:
"""
For N electrons, the wavefunction depends on 3N spatial + N spin coordinates, the electron-electron repulsion term (in Hamiltonian) couples all electrons, making the problem non-separable.
"""
 
def count_determinants(n_electrons, n_orbitals):
    """
    Count the number of Slater determinants in FCI.
    
    For N electrons in M orbitals:
    - N/2 alpha electrons can be placed in M orbitals: C(M, N/2) ways
    - N/2 beta electrons can be placed in M orbitals: C(M, N/2) ways
    - Total: C(M, N/2)² determinants
    
    This is the dimension of the FCI matrix!
    """
    from math import comb
    n_alpha = n_electrons // 2
    n_beta = n_electrons - n_alpha
    return comb(n_orbitals, n_alpha) * comb(n_orbitals, n_beta)
 
print("\n" + "="*60)
print("THE COMBINATORIAL EXPLOSION")
print("="*60)
print("\nNumber of Slater determinants for different systems:\n")
 
systems = [
    ("H₂ (2e, 4 orbitals)", 2, 4),
    ("H₂O (10e, 14 orbitals)", 10, 14),
    ("Benzene (30e, 36 orbitals)", 30, 36),
    ("Fe₂S₂ cluster (50e, 50 orbitals)", 50, 50),
    ("[4Fe-4S] cluster (100e, 76 orbitals)", 100, 76),
]
 
for name, n_e, n_orb in systems:
    n_det = count_determinants(n_e, n_orb)
    print(f"{name:40s}: {n_det:>20,.0f} determinants")
 
print("\n>>> The [4Fe-4S] cluster is Prof. Sharma's benchmark system!")
print(">>> Classical methods CANNOT solve this exactly.")
print(">>> This is why SHCI, DMRG, and AFQMC are essential.")

In [ ]:
"""
A Slater determinant is an antisymmetrized product of spin-orbitals:
 
|Φ⟩ = |χ₁χ₂...χₙ⟩ = 1/√N! |χ₁(1) χ₂(1) ... χₙ(1)|
                         |χ₁(2) χ₂(2) ... χₙ(2)|
                         |  ⋮      ⋮    ⋱    ⋮   |
                         |χ₁(N) χ₂(N) ... χₙ(N)|
 
Properties:
1. Antisymmetric under particle exchange (Pauli principle)
2. Normalized if orbitals are orthonormal
3. Not an eigenfunction of Ĥ in general (except for 1 electron)
"""
 
def slater_determinant_example():
    """
    Demonstrate Slater determinant properties with 2 electrons.
    
    For 2 electrons in orbitals φ₁ and φ₂:
    |Φ⟩ = 1/√2 [φ₁(1)φ₂(2) - φ₁(2)φ₂(1)]
    
    Using matrix form:
    |Φ⟩ = 1/√2! det|φ₁(1) φ₂(1)|
                   |φ₁(2) φ₂(2)|
    """
    print("\n" + "="*60)
    print("SLATER DETERMINANT EXAMPLE: 2 electrons")
    print("="*60)
    
    # Represent orbitals as vectors on a grid
    x = np.linspace(-3, 3, 100)
    
    # Simple Gaussian orbitals (like 1s)
    phi_1 = np.exp(-x**2)        # Orbital 1 centered at origin
    phi_2 = np.exp(-(x-1)**2)    # Orbital 2 centered at x=1
    
    # Normalize
    phi_1 = phi_1 / np.sqrt(np.trapz(phi_1**2, x))
    phi_2 = phi_2 / np.sqrt(np.trapz(phi_2**2, x))
    
    # Overlap
    S_12 = np.trapz(phi_1 * phi_2, x)
    print(f"Overlap ⟨φ₁|φ₂⟩ = {S_12:.4f}")
    print("(Non-zero overlap means these are NOT orthogonal)")
    
    # For a proper Slater determinant, we need orthonormal orbitals
    # Let's orthogonalize using Gram-Schmidt
    phi_2_orth = phi_2 - S_12 * phi_1
    phi_2_orth = phi_2_orth / np.sqrt(np.trapz(phi_2_orth**2, x))
    
    S_12_new = np.trapz(phi_1 * phi_2_orth, x)
    print(f"After orthogonalization: ⟨φ₁|φ₂'⟩ = {S_12_new:.2e}")
    
    return x, phi_1, phi_2_orth
 
x, phi_1, phi_2 = slater_determinant_example()

In [ ]:
"""
The exact wavefunction can be written as:
 
|Ψ⟩ = Σᵢ cᵢ |Dᵢ⟩
 
where |Dᵢ⟩ are Slater determinants.
 
Full CI (FCI): Include ALL determinants → Exact answer, but exponential cost
Selected CI: Include only IMPORTANT determinants → Near-exact, polynomial cost
 
SHCI (Semistochastic Heat-Bath CI) is Prof. Sharma's method for efficiently
selecting which determinants matter.
 
The key insight: Most cᵢ coefficients are tiny!
"""
 
def demonstrate_ci_sparsity():
    """
    Show that CI wavefunctions are typically sparse.
    
    This is a toy model demonstrating the principle.
    """
    print("\n" + "="*60)
    print("CI COEFFICIENT DISTRIBUTION (Toy Model)")
    print("="*60)
    
    # Simulate CI coefficients with exponential decay (typical behavior)
    np.random.seed(42)
    n_det = 10000
    
    # Most coefficients are small; few are large
    # This mimics real CI wavefunctions
    coefficients = np.random.exponential(scale=0.01, size=n_det)
    coefficients[0] = 0.9  # HF determinant dominates
    coefficients = coefficients / np.linalg.norm(coefficients)
    
    # Sort by magnitude
    sorted_coeffs = np.sort(np.abs(coefficients))[::-1]
    
    # How many determinants capture 99% of the wavefunction?
    cumsum = np.cumsum(sorted_coeffs**2)
    n_99 = np.argmax(cumsum >= 0.99) + 1
    n_999 = np.argmax(cumsum >= 0.999) + 1
    
    print(f"Total determinants: {n_det}")
    print(f"Determinants for 99% of |Ψ|²: {n_99} ({100*n_99/n_det:.2f}%)")
    print(f"Determinants for 99.9% of |Ψ|²: {n_999} ({100*n_999/n_det:.2f}%)")
    print("\n>>> This sparsity is what SHCI exploits!")
    print(">>> Prof. Sharma's SHCI efficiently finds the important determinants.")
 
demonstrate_ci_sparsity()

In [ ]:
"""
PySCF is the Python-based quantum chemistry package that Prof. Sharma 
contributed to. Let's run a simple calculation.
"""
 
print("\n" + "="*60)
print("YOUR FIRST PYSCF CALCULATION")
print("="*60)
 
try:
    from pyscf import gto, scf, fci
    
    # Define a water molecule
    mol = gto.Mole()
    mol.atom = '''
        O  0.0000  0.0000  0.1173
        H  0.0000  0.7572 -0.4692
        H  0.0000 -0.7572 -0.4692
    '''
    mol.basis = 'sto-3g'  # Minimal basis set
    mol.build()
    
    print(f"\nMolecule: Water (H₂O)")
    print(f"Basis: STO-3G (minimal)")
    print(f"Number of electrons: {mol.nelectron}")
    print(f"Number of basis functions: {mol.nao}")
    
    # Run Hartree-Fock
    mf = scf.RHF(mol)
    E_hf = mf.kernel()
    print(f"\nHartree-Fock Energy: {E_hf:.8f} Hartree")
    
    # Run Full CI (exact in this basis)
    cisolver = fci.FCI(mf)
    E_fci, ci_vec = cisolver.kernel()
    print(f"Full CI Energy: {E_fci:.8f} Hartree")
    
    # Correlation energy
    E_corr = E_fci - E_hf
    print(f"\nCorrelation Energy: {E_corr:.8f} Hartree")
    print(f"                  = {E_corr * 627.509:.4f} kcal/mol")
    
    print("\n>>> The correlation energy is what post-HF methods try to capture!")
    print(">>> For strongly correlated systems, this is MUCH larger.")
    
except ImportError:
    print("PySCF not installed. Run: pip install pyscf")
    print("Then re-run this script.")


In [ ]:
print("\n" + "="*60)
print("DAY 1 EXERCISES")
print("="*60)
print("""
1. CONCEPTUAL: 
   Why does the number of Slater determinants grow exponentially with 
   system size? Write out the formula and explain each term.
 
2. MATHEMATICAL:
   For a system with 4 electrons in 6 orbitals:
   a) How many determinants are there?
   b) What is the dimension of the FCI Hamiltonian matrix?
   c) If each matrix element takes 1 nanosecond to compute, 
      how long to build the full matrix?
 
3. PROGRAMMING:
   Modify the PySCF code above to:
   a) Calculate H₂ at different bond lengths (0.5 to 3.0 Å)
   b) Plot E_HF and E_FCI vs bond length
   c) At what bond length is correlation energy largest? Why?
 
4. READING:
   Read sections 1-3 of Chan & Sharma's DMRG review (2011).
   Write a 1-paragraph summary of what DMRG does differently than FCI.
 
5. REFLECTION:
   Prof. Sharma's SHCI can handle 30-100 active orbitals.
   How many determinants would that correspond to for 50 electrons?
   Why is this remarkable?
""")
 
 
# =============================================================================
# PART 7: KEY CONCEPTS SUMMARY
# =============================================================================
print("\n" + "="*60)
print("KEY CONCEPTS FROM DAY 1")
print("="*60)
print("""
□ The many-electron Schrödinger equation is exactly solvable only for 1 electron
□ FCI gives the exact answer but scales exponentially
□ Slater determinants are antisymmetrized products of orbitals
□ The CI wavefunction is a linear combination of determinants
□ Real CI wavefunctions are SPARSE - most coefficients are tiny
□ SHCI exploits this sparsity to achieve near-exact results efficiently
□ Correlation energy = E_exact - E_HF (what we're trying to capture)
□ Strongly correlated systems have large correlation energies
 
TOMORROW (Day 2): Deep dive into Hartree-Fock theory
- The self-consistent field procedure
- Roothaan equations
- Writing your own HF code from scratch
""")
 
if __name__ == "__main__":
    print("\n" + "="*60)
    print("Day 1 Complete! Review the exercises before moving to Day 2.")
    print("="*60)